In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3
import emoji

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [49]:
def get_html(url = 'https://www.sindipetroba.org.br/2019/noticias/page/1/?et_blo'):
    payload = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Referer": "https://google.com"
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [48]:
def converter_data_portugues(data_str):
    meses_pt_para_en = {
        'jan': 'January', 'fev': 'February', 'mar': 'March', 'abr': 'April',
        'mai': 'May', 'jun': 'June', 'jul': 'July', 'ago': 'August',
        'set': 'September', 'out': 'October', 'nov': 'November', 'dez': 'December',
        'janeiro': 'January', 'fevereiro': 'February', 'março': 'March', 'abril': 'April',
        'maio': 'May', 'junho': 'June', 'julho': 'July', 'agosto': 'August',
        'setembro': 'September', 'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'
    }

    partes = data_str.split()
    if partes[0].lower() in meses_pt_para_en:
        partes[0] = meses_pt_para_en[partes[0].lower()]
    data_convertida = ' '.join(partes)
    return datetime.strptime(data_convertida, '%B %d, %Y')


def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    locale.setlocale(locale.LC_TIME, "en_US.UTF-8") 

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        span = article.find('span', class_='published')
        date = span.text.strip()
        try:
            date = converter_data_portugues(date)
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            print(e)
            continue
        # link_date = [link, date]
        # news_links.append(link_date)


    return news_links

In [47]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [46]:
def get_next_page(fnp_url = 'https://www.sindipetroba.org.br/2019/noticias/page/', next_page_number = 1):
    validated_news_links = []
    url = fnp_url + str(next_page_number) + '/?et_blog'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [56]:
def sanitize_paragraphs(paragraphs:list, min_paragraph_len = 500, concat_trigger_size = 1500):
    '''
    sanitiza os parágrafos, realizando replace de partes de strings e concatenando parágrafos
    
    paragraphs: lista de parágrafos
    min_paragraph_len: define quais parágrafos devem ser concatenados
    concat_trigger_size: caso o tamanho da concatenação de textos esteja acima desse valor, 
                            escreve como parágrafo e reseta a variável que armazena as 
                            strings a serem concatenadas
    '''
    def append_paragraphs(current_new_paragraph):
        new_paragraphs.append(current_new_paragraph.strip())
    
    paragraphs = [emoji.demojize(paragraph) for paragraph in [paragraph\
                                                              .replace('\xa0',' ')\
                                                              .replace('\n',' ')\
                                                              .replace('\t',' ')\
                                                              .replace('[email-protected]', '')\
                                                              .strip() \
                                                              for paragraph in paragraphs] \
                  if len(paragraph)>0] #removendo emoji, tratanto texto e realizando strip para então construir lista de parágrafos que tenham len > 0
    paragraphs_mask = [len(paragraph) < min_paragraph_len for paragraph in paragraphs] # true são os abaixo do min_paragraph_len, precisarao ser tratados
    if any(paragraphs_mask): #caso algum elemento precise ser tratado, trigga processo
        new_paragraphs = []
        current_new_paragraph = ''
        for i in range(len(paragraphs)):
            if len(current_new_paragraph) >= concat_trigger_size:
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
            if paragraphs_mask[i]: #caso seja menor que o min_paragraph_len
                current_new_paragraph = current_new_paragraph + ' ' + paragraphs[i]
                if i == len(paragraphs)-1: #caso seja o ultimo elemento
                    append_paragraphs(current_new_paragraph)
            else:
                current_new_paragraph = current_new_paragraph + '' + paragraphs[i]
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
                    
    return new_paragraphs

In [57]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find_all('p')

    paragraphs_text = [p.get_text(strip=True) for p in paragraphs]
    paragraphs = sanitize_paragraphs(paragraphs_text)

    return title, paragraphs

In [58]:
def main():
    next_page_number = 1
    validated_news_links = []
    url_default = 'https://www.sindipetroba.org.br/2019/noticias/page/'
    url = url_default + str(next_page_number) + '/?et_blog'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(url_default, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'BA',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [59]:
result = main()
print(len(result))
result

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 69/69 [02:06<00:00,  1.84s/it]

254


[{'sindicato': 'BA',
  'url': 'https://www.sindipetroba.org.br/2019/sindipetro-cobra-acelen-por-perseguicoes-no-ssma-e-exige-medidas-para-garantir-seguranca-dos-trabalhadores/',
  'titulo': '\nSindipetro cobra Acelen por perseguições no SSMA e exige medidas para garantir segurança dos trabalhadores\n',
  'data': datetime.datetime(2025, 8, 21, 0, 0),
  'paragrafo': 'agosto 21, 2025 |Categoria:Banner Principal,Notícia Na manhã desta quinta-feira (21), diretores do Sindipetro Bahia e representantes do setor de Recursos Humanos da Acelen se reuniram virtualmente para tratar das graves denúncias envolvendo o setor de Segurança, Saúde e Meio Ambiente (SSMA) da Refinaria gerida pela empresa. O sindicato cobrou explicações sobre a demissão de um trabalhador logo após a “roda de conversa” promovida pela empresa, episódio interpretado pela categoria como um ato de perseguição e intimidação, que reforça o clima de insegurança denunciado pelos empregados. Os representantes do RH negaram qualquer r